# Topic Modeling of Recent Academic Publications using BERTopic  
## Discovering Hot Research Topics from 2020 to 2025

In [ ]:
%%capture
!pip install bertopic
!pip install umap-learn
!pip install hdbscan
!pip install sentence-transformers
!pip install plotly

In [ ]:
!pip install -qq numpy==1.26.4 gensim
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/MachineLearning/processed.csv")
print(df.shape)
df.head()

In [ ]:
df = df.dropna(subset=['combined_lemmatized'])
documents = df['combined_lemmatized'].tolist()
years = df["year"].astype(str).tolist()
print(f"Total documents: {len(documents)}")
print("First document sample:")
print(documents[0][:300], "...")

##2. Modeling

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

### 2.1. Vectorizer Model

In [ ]:
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    stop_words="english",
    min_df=5
)

### 2.2. Embedding Model

In [ ]:
embedding_model = SentenceTransformer("paraphrase-mpnet-base-v2")

### 2.3. BERTopic Model

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=50,
    calculate_probabilities=False,
    verbose=True
)

#### 2.3.1. Training

In [ ]:
import time

start = time.time()
topics, probs = topic_model.fit_transform(documents)
end = time.time()
duration = end - start
print(f"Training completed in {duration:.2f} seconds.")

In [ ]:
save_path = "/content/drive/MyDrive/bertopic_model"
topic_model.save(save_path)
print("Model saved to Google Drive.")

In [ ]:
topic_model.get_topic_info()

## 3. Visualizations

In [ ]:
fig = topic_model.visualize_documents(documents)

for i in range(len(fig.data)):
    fig.data[i].text = None

fig.show()

In [ ]:
topic_model.visualize_topics()

In [ ]:
topics_over_time = topic_model.topics_over_time(documents, years)
topic_model.visualize_topics_over_time(topics_over_time)

### Topics over Time - Growing Topics from 2020 to 2025

In [ ]:
topics_over_time["Year"] = pd.to_datetime(topics_over_time["Timestamp"]).dt.year.astype(str)
pivot = topics_over_time.pivot_table(
    index="Year", columns="Topic", values="Frequency", aggfunc="sum"
)

first_year = pivot.index.min()
last_year = pivot.index.max()
growth = pivot.loc[last_year] - pivot.loc[first_year]

top_growing_topics = growth.sort_values(ascending=False).head(10).index.tolist()

In [ ]:
for topic_id in top_growing_topics:
    words = topic_model.get_topic(topic_id)
    keywords = [word for word, _ in words]
    print(f"Topic {topic_id}: {', '.join(keywords)}")

#### Growing Topics from 2020 to 2025


| Topic ID | Description                               |
|----------|-------------------------------------------------|
| 5        | Diffusion-Based Image & Video Generation         |
| 21       | Reasoning & Prompting in LLMs                     |
| 19       | Multimodal Question Answering (VQA)               |
| 54       | 3D Rendering via Gaussian Splatting                |
| 18       | Code Generation & Software Debugging               |
| 35       | Retrieval-Augmented Generation (RAG)               |
| 36       | Video Understanding & Captioning                    |
| 91       | Human Motion Synthesis & Animation                   |
| 64       | Vision Transformers (ViTs)                            |
| 113      | Parameter-Efficient Fine-Tuning (LoRA, PEFT)          |

In [ ]:
topic_model.visualize_hierarchy()

In [ ]:
topic_model.visualize_heatmap()

## 4. Performance Metrics

### 4.1. Coherence Scores

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

tokenized_docs = [doc.split() for doc in documents]
dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

topics = []
for topic_num in topic_model.get_topics().keys():
    topic = topic_model.get_topic(topic_num)
    if topic is not None:
        words = [word for word, _ in topic]
        topics.append(words)

coherence_cv = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='c_v').get_coherence()
coherence_umass = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='u_mass').get_coherence()
coherence_npmi = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='c_npmi').get_coherence()
coherence_uci = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='c_uci').get_coherence()

print(f"C_v Coherence:     {coherence_cv:.4f}")
print(f"U_Mass Coherence: {coherence_umass:.4f}")
print(f"NPMI Coherence:   {coherence_npmi:.4f}")
print(f"UCI Coherence:    {coherence_uci:.4f}")

### 4.2. PUW

In [ ]:
all_words = [word for topic in topics for word in topic]

unique_words = set(all_words)
puw = len(unique_words) / len(all_words)

print(f"Proportion of Unique Words (PUW): {puw:.4f}")

### 4.3. Avg. Jaccard Similarity

In [ ]:
from itertools import combinations

def jaccard_similarity(set1, set2):
    return len(set1 & set2) / len(set1 | set2)

jaccard_scores = []
for t1, t2 in combinations(topics, 2):
    jaccard_scores.append(jaccard_similarity(set(t1), set(t2)))

avg_jaccard = sum(jaccard_scores) / len(jaccard_scores)
print(f"Average Jaccard Similarity between topics: {avg_jaccard:.4f}")